## Pengaturan konfigurasi awal

#### 1. Impor modul yang dibutuhkan

In [17]:
import os
import re
import pandas as pd
from openpyxl import load_workbook

#### 2. Membuat fungsi pendukung

In [18]:
# Membaca data dari file excel
def read_data_from_excel(file_path, sheet_name):
    # Baca sheet dengan header baris 1-3
    df = pd.read_excel(file_path, sheet_name=sheet_name, header=[0,1,2])

    # Gabungkan header multi-level jadi single-level
    df.columns = [
        "_".join([str(c).strip() for c in col if "Unnamed" not in str(c)])
            .lower()
            .replace(" ", "_")
        for col in df.columns.values
    ]

    # Hapus double underscore kalau ada
    df.columns = [c.replace("__", "_").strip("_") for c in df.columns]

    # Cek hasil kolom
    print(df.columns.tolist())

    # Tampilkan data
    print(df.head())

    return df

# Melakukan cleaning data pada dataframe
def cleaning_data(df):
    # bikin copy biar df asli nggak berubah
    df_clean = df.copy()

    # ambil semua kolom object kecuali 'kabupaten/kota'
    object_cols = [col for col in df_clean.select_dtypes(include=["object"]).columns if col != "kabupaten/kota" and col != "provinsi"]

    # convert semua kolom object itu jadi numeric
    # kalau ada "-", "" atau value non-numeric lain -> otomatis jadi NaN
    for col in object_cols:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

    # cek hasil konversi
    print(df_clean.dtypes)

    return df_clean

# Mendapatkan data pada periode triwulan berjalan
def get_unpaired_columns(columns):
    # pasangan yang valid
    paired_suffixes = {"ril", "rev", "prov", "kabkot"}

    unpaired = []

    for col in columns:
        # ambil bagian suffix (setelah underscore paling belakang)
        suffix = col.split("_")[-1]

        # cek apakah suffix masuk pasangan
        if suffix not in paired_suffixes:
            unpaired.append(col)

    return unpaired

# Menggabungkan data evaluasi
def concat_evaluasi(list_evaluasi):
    return pd.concat(list_evaluasi, ignore_index=True)    

# Memastikan nama sheet valid
def safe_sheet_name(name):
    return re.sub(r'[:\\/*?\[\]]', "_", str(name))[:31]

# Menulis data ke dalam file excel
def write_to_excel(df, group_col):
    for group_value in df[group_col].unique():
        df_group = df[df[group_col] == group_value]
        kode = str(df_group["kode"].iloc[0])  # ambil kode group_col

        # Nama file sesuai kode group_col
        if group_col == 'provinsi':
            output_file = f"data/output/evaluasi_{kode[:2]}.xlsx"
            sheet_name = safe_sheet_name(kode[:2] + "00")
        elif group_col == 'kabupaten/kota':
            output_file = f"data/output/evaluasi_{kode[:2]}.xlsx"
            sheet_name = safe_sheet_name(kode[:4])
        

        if os.path.exists(output_file):
            # Load workbook lama
            book = load_workbook(output_file)

            if sheet_name in book.sheetnames:
                # Sheet sudah ada → cari baris terakhir
                ws = book[sheet_name]
                startrow = ws.max_row  # tulis setelah baris terakhir

                with pd.ExcelWriter(output_file, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
                    df_group.to_excel(writer, sheet_name=sheet_name, index=False, header=False, startrow=startrow)
                print(f"Data {group_value} ditambahkan ke sheet {sheet_name} di {output_file}")

            else:
                # Sheet belum ada → tambahkan sheet baru
                with pd.ExcelWriter(output_file, engine="openpyxl", mode="a") as writer:
                    df_group.to_excel(writer, sheet_name=sheet_name, index=False)
                print(f"Sheet {sheet_name} ditambahkan ke {output_file}")

        else:
            # File belum ada → buat baru
            with pd.ExcelWriter(output_file, engine="openpyxl", mode="w") as writer:
                df_group.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"File baru dibuat: {output_file}, dengan sheet {sheet_name}")


#### 3. Menentukan lokasi file yang akan digunakan sebagai input data

In [19]:
file_path = os.path.join("data/input", "Evaluasi.xlsx")

## Evaluasi Diskrepansi Prov vs KabKot

#### 1. Membaca file Evaluasi.xlsx pada sheet Prov vs KabKot

In [20]:
# Membaca data dari file excel (Diskrepansi 2024, Diskrepansi 2023)
df = read_data_from_excel(file_path, "Diskrepansi 2023")

# Cleaning data
df_clean = cleaning_data(df)
df_clean.info()

['kode', 'provinsi', 'adhb_pdrb_q1-23', 'adhb_pdrb_q2-23', 'adhb_pdrb_q3-23', 'adhb_pdrb_q4-23', 'adhb_pdrb_2023', 'adhb_pkrt_q1-23', 'adhb_pkrt_q2-23', 'adhb_pkrt_q3-23', 'adhb_pkrt_q4-23', 'adhb_pkrt_2023', 'adhb_pklnprt_q1-23', 'adhb_pklnprt_q2-23', 'adhb_pklnprt_q3-23', 'adhb_pklnprt_q4-23', 'adhb_pklnprt_2023', 'adhb_pkp_q1-23', 'adhb_pkp_q2-23', 'adhb_pkp_q3-23', 'adhb_pkp_q4-23', 'adhb_pkp_2023', 'adhb_pmtb_q1-23', 'adhb_pmtb_q2-23', 'adhb_pmtb_q3-23', 'adhb_pmtb_q4-23', 'adhb_pmtb_2023', 'adhb_pi_q1-23', 'adhb_pi_q2-23', 'adhb_pi_q3-23', 'adhb_pi_q4-23', 'adhb_pi_2023', 'adhb_net_ekspor_q1-23', 'adhb_net_ekspor_q2-23', 'adhb_net_ekspor_q3-23', 'adhb_net_ekspor_q4-23', 'adhb_net_ekspor_2023', 'adhk_pdrb_q1-23', 'adhk_pdrb_q2-23', 'adhk_pdrb_q3-23', 'adhk_pdrb_q4-23', 'adhk_pdrb_2023', 'adhk_pkrt_q1-23', 'adhk_pkrt_q2-23', 'adhk_pkrt_q3-23', 'adhk_pkrt_q4-23', 'adhk_pkrt_2023', 'adhk_pklnprt_q1-23', 'adhk_pklnprt_q2-23', 'adhk_pklnprt_q3-23', 'adhk_pklnprt_q4-23', 'adhk_pklnprt_2

#### 2. Evaluasi diskrepansi ADHB dan ADHK

In [21]:
# 1. Ambil kolom yang diawali adhb atau adhk
target_cols = [col for col in df_clean.columns if col.startswith("adhb") or col.startswith("adhk")]

# 2. Buat list untuk menampung hasil diskrepansi
evaluasi_diskrepansi = []

# 3. Loop tiap kolom target
for col in target_cols:
    for idx, val in df_clean[col].items():
        # PDRB
        if "pdrb" in col:
            # ADHB dan ADHK Q1 & Q2 2025
            if any(q in col for q in ["q1", "q2", "q3", "q4"]):
                if pd.notna(val) and (val < -5 or val > 5):
                    # Pisahkan nama kolom jadi komponen
                    # contoh: adhb_pdrb_q1_25 → ['adhb', 'pdrb', 'q1', '25']
                    parts = col.split("_")
                    
                    periode = parts[-1]  # ambil q1 / q2

                    # tentukan evaluasi berdasarkan isi col
                    if "adhb" in col:
                        evaluasi_text = "Diskrepansi ADHB antara Provinsi dan total Kabupaten/Kota lebih dari (+-5%)"
                    elif "adhk" in col:
                        evaluasi_text = "Diskrepansi ADHK antara Provinsi dan total Kabupaten/Kota lebih dari (+-5%)"

                    record = {
                        "kode": df_clean.at[idx, "kode"],
                        "provinsi": df_clean.at[idx, "provinsi"],
                        "periode": periode,
                        "nilai": val,
                        "kolom": col,  # tambahan biar tau sumbernya
                        "evaluasi": evaluasi_text
                    }
                    evaluasi_diskrepansi.append(record)

            # ADHB dan ADHK Q3 & Q4 2025
            # elif "q3" in col:
            # elif any(q in col for q in ["q3", "q4"]):
            #     if pd.notna(val) and (val < -3 or val > 3):
            #         # Pisahkan nama kolom jadi komponen
            #         # contoh: adhb_pdrb_q1_25 → ['adhb', 'pdrb', 'q1', '25']
            #         parts = col.split("_")
                    
            #         periode = parts[-1]  # ambil q1 / q2

            #         # tentukan evaluasi berdasarkan isi col
            #         if "adhb" in col:
            #             evaluasi_text = "Diskrepansi ADHB antara Provinsi dan total Kabupaten/Kota lebih dari (+-3%)"
            #         elif "adhk" in col:
            #             evaluasi_text = "Diskrepansi ADHK antara Provinsi dan total Kabupaten/Kota lebih dari (+-3%)"

            #         record = {
            #             "kode": df_clean.at[idx, "kode"],
            #             "provinsi": df_clean.at[idx, "provinsi"],
            #             "periode": periode,
            #             "nilai": val,
            #             "kolom": col,  # tambahan biar tau sumbernya
            #             "evaluasi": evaluasi_text
            #         }
            #         evaluasi_diskrepansi.append(record)
            
            # ADHB dan ADHK Tahunan 2025
            elif "Q" not in col:
                if pd.notna(val) and (val < -5 or val > 5):
                    # Pisahkan nama kolom jadi komponen
                    # contoh: adhb_pdrb_2025 → ['adhb', 'pdrb', '2025']
                    parts = col.split("_")
                    
                    periode = parts[-1]  # ambil tahun

                    # tentukan evaluasi berdasarkan isi col
                    if "adhb" in col:
                        evaluasi_text = "Diskrepansi ADHB antara Provinsi dan total Kabupaten/Kota lebih dari (+-5%)"
                    elif "adhk" in col:
                        evaluasi_text = "Diskrepansi ADHK antara Provinsi dan total Kabupaten/Kota lebih dari (+-5%)"

                    record = {
                        "kode": df_clean.at[idx, "kode"],
                        "provinsi": df_clean.at[idx, "provinsi"],
                        "periode": periode,
                        "nilai": val,
                        "kolom": col,  # tambahan biar tau sumbernya
                        "evaluasi": evaluasi_text
                    }
                    evaluasi_diskrepansi.append(record)
                    
        # Komponen
        elif "pdrb" not in col:
            # ADHB dan ADHK Q1, Q2, Q3 dan Q4 2025
            if pd.notna(val) and (val < -5 or val > 5):
                # Pisahkan nama kolom jadi komponen
                # contoh: adhb_pdrb_q1_25 → ['adhb', 'pdrb', 'q1', '25']
                parts = col.split("_")
                
                periode = parts[-1]  # ambil q1 / q2

                # tentukan evaluasi berdasarkan isi col
                if "adhb" in col:
                    evaluasi_text = "Diskrepansi ADHB antara Provinsi dan total Kabupaten/Kota lebih dari (+-5%)"
                elif "adhk" in col:
                    evaluasi_text = "Diskrepansi ADHK antara Provinsi dan total Kabupaten/Kota lebih dari (+-5%)"
                    
                record = {
                    "kode": df_clean.at[idx, "kode"],
                    "provinsi": df_clean.at[idx, "provinsi"],
                    "periode": periode,
                    "nilai": val,
                    "kolom": col,  # tambahan biar tau sumbernya
                    "evaluasi": evaluasi_text
                }
                evaluasi_diskrepansi.append(record)
                
# 4. Ubah jadi DataFrame biar enak lihat
evaluasi_diskrepansi = pd.DataFrame(evaluasi_diskrepansi)
print(evaluasi_diskrepansi)

     kode          provinsi periode       nilai                 kolom  \
0      72   Sulawesi Tengah   q1-23    5.536140        adhb_pkp_q1-23   
1      21    Kepulauan Riau   q2-23   -7.663545        adhb_pkp_q2-23   
2      14              Riau   q4-23    6.898591        adhb_pkp_q4-23   
3      91       Papua Barat   q4-23   -5.963761        adhb_pkp_q4-23   
4      72   Sulawesi Tengah   q4-23   -5.539989       adhb_pmtb_q4-23   
..    ...               ...     ...         ...                   ...   
226    61  Kalimantan Barat    2023  -65.027902  adhk_net_ekspor_2023   
227    71    Sulawesi Utara    2023   23.979611  adhk_net_ekspor_2023   
228    73  Sulawesi Selatan    2023   42.218210  adhk_net_ekspor_2023   
229    75         Gorontalo    2023    6.566724  adhk_net_ekspor_2023   
230    82      Maluku Utara    2023  126.860792  adhk_net_ekspor_2023   

                                              evaluasi  
0    Diskrepansi ADHB antara Provinsi dan total Kab...  
1    Disk

#### 3. Evaluasi distribusi

In [22]:
# 1. Ambil semua kolom distribusi
dist_cols = [col for col in df_clean.columns if col.startswith("distribusi")]

# 2. Buat base name tanpa suffix `.1`
pairs = {}
for col in dist_cols:
    # base = re.sub(r'\.1$', '', col)  # buang .1 di akhir
    base = col.replace("p-", "").replace("k-", "")
    pairs.setdefault(base, []).append(col)

# 3. Evaluasi distribusi
evaluasi_dist = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        # col0 = [c for c in cols if not c.endswith(".1")][0]
        # col1 = [c for c in cols if c.endswith(".1")][0]
        col0 = [c for c in cols if ("p-") in c][0]
        col1 = [c for c in cols if ("k-") in c][0]

        # Ambil periode (contoh: distribusi_pdrb_q1-25 → q1)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df_clean.index:
            v0 = df_clean.at[idx, col0]
            v1 = df_clean.at[idx, col1]

            # Distribusi PDRB & Komponen Q1,Q2,Q3 2025
            if pd.notna(v0) and pd.notna(v1):
                selisih = abs(v0 - v1)
                if selisih > 5:
                    record = {
                        "kode": df_clean.at[idx, "kode"],
                        "provinsi": df_clean.at[idx, "provinsi"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Diskrepansi Distribusi antara Provinsi dan total Kabupaten/Kota lebih dari (+-5 point)"
                    }
                    evaluasi_dist.append(record)

# 4. Jadi DataFrame
evaluasi_dist = pd.DataFrame(evaluasi_dist)
print(evaluasi_dist)

Empty DataFrame
Columns: []
Index: []


#### 4. Evaluasi growth Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y 

In [23]:
# 1. Ambil semua kolom Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y 
growth_cols = [col for col in df_clean.columns if col.startswith("q-to-q") or col.startswith("y-on-y") or col.startswith("c-to-c") or col.startswith("implisit_q-to-q") or col.startswith("implisit_y-on-y")]

# 2. Buat base name tanpa suffix `.1`
pairs = {}
for col in growth_cols:
    # base = re.sub(r'\.1$', '', col)  # buang .1 di akhir
    base = col.replace("p-", "").replace("k-", "")
    pairs.setdefault(base, []).append(col)

# 3. Evaluasi Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y
evaluasi_growth = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        # col0 = [c for c in cols if not c.endswith(".1")][0]
        # col1 = [c for c in cols if c.endswith(".1")][0]
        col0 = [c for c in cols if ("p-") in c][0]
        col1 = [c for c in cols if ("k-") in c][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df_clean.index:
            v0 = df_clean.at[idx, col0]
            v1 = df_clean.at[idx, col1]
            
            # Growth Q-to-Q
            if 'q-to-q' in col0 and 'implisit' not in col0:
                # PDRB
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' in col0:
                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                        }
                        evaluasi_growth.append(record)
                    elif selisih > 0.5:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Diskrepansi growth Q-to-Q antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,5 point)"
                        }
                        evaluasi_growth.append(record)
                # Komponen
                elif pd.notna(v0) and pd.notna(v1) and 'pdrb' not in col0:
                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                        }
                        evaluasi_growth.append(record)
                    elif selisih > 1:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Diskrepansi growth Q-to-Q antara Provinsi dan total Kabupaten/Kota lebih dari (+-1 point)"
                        }
                        evaluasi_growth.append(record)

            # Growth Y-on-Y
            elif "y-on-y" in col0 and "implisit" not in col0:
                # PDRB
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' in col0:
                    selisih = abs(v1 - v0)
                    # Q1-Q4 2025
                    if any(q in col0 for q in ["q1", "q2", "q3", "q4"]):
                        if v0 * v1 < 0:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                            }
                            evaluasi_growth.append(record)
                        elif selisih > 0.5:
                            print(selisih)
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "Diskrepansi growth Y-on-Y antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,5 point)"
                            }
                            evaluasi_growth.append(record)
                    # Q3 dan Q4 2025
                    # elif "q3" in col0:
                    # if any(q in col0 for q in ["q3", "q4"]):
                    #     if v0 * v1 < 0:
                    #         record = {
                    #             "kode": df_clean.at[idx, "kode"],
                    #             "provinsi": df_clean.at[idx, "provinsi"],
                    #             "periode": periode,
                    #             "nilai": v1,
                    #             "kolom": col0 + " vs " + col1,
                    #             "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                    #         }
                    #         evaluasi_growth.append(record)
                    #     elif selisih > 0.01:
                    #         record = {
                    #             "kode": df_clean.at[idx, "kode"],
                    #             "provinsi": df_clean.at[idx, "provinsi"],
                    #             "periode": periode,
                    #             "nilai": v1,
                    #             "kolom": col0 + " vs " + col1,
                    #             "evaluasi": "Diskrepansi growth Y-on-Y antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,01 point)"
                    #         }
                    #         evaluasi_growth.append(record)
                # Komponen
                elif pd.notna(v0) and pd.notna(v1) and 'pdrb' not in col0:
                    selisih = abs(v1 - v0)
                    # Q1-Q4 2025
                    if any(q in col0 for q in ["q1", "q2", "q3", "q4"]):
                        if v0 * v1 < 0:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                            }
                            evaluasi_growth.append(record)
                        elif selisih > 1:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "Diskrepansi growth Y-on-Y antara Provinsi dan total Kabupaten/Kota lebih dari (+-1 point)"
                            }
                            evaluasi_growth.append(record)
                    # Q3 2025
                    # elif "q3" in col0:
                    #     if v0 * v1 < 0:
                    #         record = {
                    #             "kode": df_clean.at[idx, "kode"],
                    #             "provinsi": df_clean.at[idx, "provinsi"],
                    #             "periode": periode,
                    #             "nilai": v1,
                    #             "kolom": col0 + " vs " + col1,
                    #             "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                    #         }
                    #         evaluasi_growth.append(record)
                    #     elif selisih > 0.5:
                    #         record = {
                    #             "kode": df_clean.at[idx, "kode"],
                    #             "provinsi": df_clean.at[idx, "provinsi"],
                    #             "periode": periode,
                    #             "nilai": v1,
                    #             "kolom": col0 + " vs " + col1,
                    #             "evaluasi": "Diskrepansi growth Y-on-Y antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,5 point)"
                    #         }
                    #         evaluasi_growth.append(record)
                
            # Growth C-to-C
            elif 'c-to-c' in col0:
                # PDRB
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' in col0:
                    selisih = abs(v1 - v0)
                    # Q1-Q3 2025
                    if any(q in col0 for q in ["q1", "q2", "q3", "q4"]):
                        if v0 * v1 < 0:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                            }
                            evaluasi_growth.append(record)
                        elif selisih > 0.5:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "provinsi": df_clean.at[idx, "provinsi"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "Diskrepansi growth C-to-C antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,5 point)"
                            }
                            evaluasi_growth.append(record)
                    # Q4 2025
                    # elif any(q in col0 for q in ["q4"]):
                    #     if v0 * v1 < 0:
                    #         record = {
                    #             "kode": df_clean.at[idx, "kode"],
                    #             "provinsi": df_clean.at[idx, "provinsi"],
                    #             "periode": periode,
                    #             "nilai": v1,
                    #             "kolom": col0 + " vs " + col1,
                    #             "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                    #         }
                    #         evaluasi_growth.append(record)
                    #     elif selisih > 0.02:
                    #         record = {
                    #             "kode": df_clean.at[idx, "kode"],
                    #             "provinsi": df_clean.at[idx, "provinsi"],
                    #             "periode": periode,
                    #             "nilai": v1,
                    #             "kolom": col0 + " vs " + col1,
                    #             "evaluasi": "Diskrepansi growth C-to-C antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,02 point)"
                    #         }
                    #         evaluasi_growth.append(record)
                # Komponen
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' not in col0:
                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                        }
                        evaluasi_growth.append(record)
                    elif selisih > 1:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Diskrepansi growth C-to-C antara Provinsi dan total Kabupaten/Kota lebih dari (+-1 point)"
                        }
                        evaluasi_growth.append(record)

            # Growth Implisit Q-to-Q 
            elif 'implisit_q-to-q' in col0:
                if pd.notna(v0) and pd.notna(v1):
                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                        }
                        evaluasi_growth.append(record)
                    elif selisih > 0.5:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Diskrepansi Implisit Q-to-Q antara Provinsi dan total Kabupaten/Kota lebih dari (+-0,5 point)"
                        }
                        evaluasi_growth.append(record)

            # Growth Implisit Y-on-Y 
            elif 'implisit_y-on-y' in col0:
                if pd.notna(v0) and pd.notna(v1):
                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai antara Provinsi dan total Kabupaten/Kota beda arah"
                        }
                        evaluasi_growth.append(record)
                    elif selisih > 1:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "provinsi": df_clean.at[idx, "provinsi"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Diskrepansi Implisit Y-on-Y antara Provinsi dan total Kabupaten/Kota lebih dari (+-1 point)"
                        }
                        evaluasi_growth.append(record)

# 4. Jadi DataFrame
evaluasi_growth = pd.DataFrame(evaluasi_growth)
print(evaluasi_growth)

0.6402524087595642
0.9796529421924558
1.2586215333116062
0.7121728308071305
0.724346818173832
2.126298695426593
0.6692994753082786
1.9630054121860088
0.5006977749949604
0.6006319554561932
1.1125911821697088
3.1906382801179554
0.6467532992677558
1.1566499493051712
0.5024904455672914
0.6134917122269528
1.589810806562319
0.7784475974768155
0.6200190635989067
0.6010381475202737
0.8061912384199879
1.000785443509006
1.407941205605336
0.6061189224038182
0.9028045889052887
0.5080990012072988
0.8969991179829062
1.5568405427333278
0.5937736223633197
0.5184354930081865
1.9065157171395892
0.8688705211983656
1.5122636850307614
1.2580592698748383
     kode            provinsi periode     nilai  \
0      16    Sumatera Selatan   q1-23 -1.350870   
1      62   Kalimantan Tengah   q1-23 -4.031682   
2      63  Kalimantan Selatan   q1-23 -4.772017   
3      72     Sulawesi Tengah   q1-23 -3.415249   
4      73    Sulawesi Selatan   q1-23 -4.643299   
..    ...                 ...     ...       ...   
95

## Menggabungkan seluruh hasil evaluasi

In [24]:
evaluasi = concat_evaluasi([evaluasi_diskrepansi, evaluasi_dist, evaluasi_growth])
print(evaluasi.head())

   kode         provinsi periode     nilai            kolom  \
0    72  Sulawesi Tengah   q1-23  5.536140   adhb_pkp_q1-23   
1    21   Kepulauan Riau   q2-23 -7.663545   adhb_pkp_q2-23   
2    14             Riau   q4-23  6.898591   adhb_pkp_q4-23   
3    91      Papua Barat   q4-23 -5.963761   adhb_pkp_q4-23   
4    72  Sulawesi Tengah   q4-23 -5.539989  adhb_pmtb_q4-23   

                                            evaluasi  
0  Diskrepansi ADHB antara Provinsi dan total Kab...  
1  Diskrepansi ADHB antara Provinsi dan total Kab...  
2  Diskrepansi ADHB antara Provinsi dan total Kab...  
3  Diskrepansi ADHB antara Provinsi dan total Kab...  
4  Diskrepansi ADHB antara Provinsi dan total Kab...  


## Menuliskan hasil evaluasi kedalam file excel

In [25]:
write_to_excel(evaluasi, "provinsi")

File baru dibuat: data/output/evaluasi_72.xlsx, dengan sheet 7200
File baru dibuat: data/output/evaluasi_21.xlsx, dengan sheet 2100
File baru dibuat: data/output/evaluasi_14.xlsx, dengan sheet 1400
File baru dibuat: data/output/evaluasi_91.xlsx, dengan sheet 9100
File baru dibuat: data/output/evaluasi_11.xlsx, dengan sheet 1100
File baru dibuat: data/output/evaluasi_15.xlsx, dengan sheet 1500
File baru dibuat: data/output/evaluasi_17.xlsx, dengan sheet 1700
File baru dibuat: data/output/evaluasi_18.xlsx, dengan sheet 1800
File baru dibuat: data/output/evaluasi_34.xlsx, dengan sheet 3400
File baru dibuat: data/output/evaluasi_61.xlsx, dengan sheet 6100
File baru dibuat: data/output/evaluasi_73.xlsx, dengan sheet 7300
File baru dibuat: data/output/evaluasi_62.xlsx, dengan sheet 6200
File baru dibuat: data/output/evaluasi_16.xlsx, dengan sheet 1600
File baru dibuat: data/output/evaluasi_12.xlsx, dengan sheet 1200
File baru dibuat: data/output/evaluasi_31.xlsx, dengan sheet 3100
File baru 

## Evaluasi Revisi & Growth

#### 1. Membaca file Evaluasi.xlsx pada sheet Revisi & Growth

In [26]:
# Membaca data dari file excel (Revisi 2024, Revisi 2023)
df = read_data_from_excel(file_path, "Revisi 2023")

# Cleaning data
df_clean = cleaning_data(df)
df_clean.info()

['kode', 'kabupaten/kota', 'adhb_pdrb_q1-23_ril', 'adhb_pdrb_q1-23_rev', 'adhb_pdrb_q2-23_ril', 'adhb_pdrb_q2-23_rev', 'adhb_pdrb_q3-23_ril', 'adhb_pdrb_q3-23_rev', 'adhb_pdrb_q4-23_ril', 'adhb_pdrb_q4-23_rev', 'adhb_pdrb_2023_ril', 'adhb_pdrb_2023_rev', 'adhb_pkrt_q1-23_ril', 'adhb_pkrt_q1-23_rev', 'adhb_pkrt_q2-23_ril', 'adhb_pkrt_q2-23_rev', 'adhb_pkrt_q3-23_ril', 'adhb_pkrt_q3-23_rev', 'adhb_pkrt_q4-23_ril', 'adhb_pkrt_q4-23_rev', 'adhb_pkrt_2023_ril', 'adhb_pkrt_2023_rev', 'adhb_pklnprt_q1-23_ril', 'adhb_pklnprt_q1-23_rev', 'adhb_pklnprt_q2-23_ril', 'adhb_pklnprt_q2-23_rev', 'adhb_pklnprt_q3-23_ril', 'adhb_pklnprt_q3-23_rev', 'adhb_pklnprt_q4-23_ril', 'adhb_pklnprt_q4-23_rev', 'adhb_pklnprt_2023_ril', 'adhb_pklnprt_2023_rev', 'adhb_pkp_q1-23_ril', 'adhb_pkp_q1-23_rev', 'adhb_pkp_q2-23_ril', 'adhb_pkp_q2-23_rev', 'adhb_pkp_q3-23_ril', 'adhb_pkp_q3-23_rev', 'adhb_pkp_q4-23_ril', 'adhb_pkp_q4-23_rev', 'adhb_pkp_2023_ril', 'adhb_pkp_2023_rev', 'adhb_pmtb_q1-23_ril', 'adhb_pmtb_q1-23_r

#### 2. Evaluasi revisi ADHB dan ADHK

In [27]:
# 1. Ambil semua kolom ADHB dan ADHK 
revisi_harga_cols = [col for col in df.columns if col.startswith("adhb") or col.startswith("adhk")]
print(revisi_harga_cols)
# 2. Buat base name tanpa suffix `ril/rev`
pairs = {}
for col in revisi_harga_cols:
    if "ril" in col or "rev" in col:
        base = "_".join(col.split("_")[:-1])  # buang suffix ril/rev
        pairs.setdefault(base, []).append(col)

# 3. Evaluasi ADHB dan ADHK
evaluasi_revisi_harga = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if c.endswith("_ril")][0]
        col1 = [c for c in cols if c.endswith("_rev")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df.index:
            v0 = df.at[idx, col0]
            v1 = df.at[idx, col1]

            if pd.notna(v0) and pd.notna(v1) and v1 != 0:
                selisih = abs((v1 - v0) / v0)
                if v0 * v1 < 0:
                    record = {
                        "kode": df.at[idx, "kode"],
                        "kabupaten/kota": df.at[idx, "kabupaten/kota"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Nilai revisi dan nilai rilis beda arah"
                    }
                    evaluasi_revisi_harga.append(record)
                elif selisih > 0.3:
                    record = {
                        "kode": df.at[idx, "kode"],
                        "kabupaten/kota": df.at[idx, "kabupaten/kota"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Nilai revisi lebih dari (+-30%) dari nilai rilis"
                    }
                    evaluasi_revisi_harga.append(record)

# 4. Jadi DataFrame
evaluasi_revisi_harga = pd.DataFrame(evaluasi_revisi_harga)
print(evaluasi_revisi_harga)

['adhb_pdrb_q1-23_ril', 'adhb_pdrb_q1-23_rev', 'adhb_pdrb_q2-23_ril', 'adhb_pdrb_q2-23_rev', 'adhb_pdrb_q3-23_ril', 'adhb_pdrb_q3-23_rev', 'adhb_pdrb_q4-23_ril', 'adhb_pdrb_q4-23_rev', 'adhb_pdrb_2023_ril', 'adhb_pdrb_2023_rev', 'adhb_pkrt_q1-23_ril', 'adhb_pkrt_q1-23_rev', 'adhb_pkrt_q2-23_ril', 'adhb_pkrt_q2-23_rev', 'adhb_pkrt_q3-23_ril', 'adhb_pkrt_q3-23_rev', 'adhb_pkrt_q4-23_ril', 'adhb_pkrt_q4-23_rev', 'adhb_pkrt_2023_ril', 'adhb_pkrt_2023_rev', 'adhb_pklnprt_q1-23_ril', 'adhb_pklnprt_q1-23_rev', 'adhb_pklnprt_q2-23_ril', 'adhb_pklnprt_q2-23_rev', 'adhb_pklnprt_q3-23_ril', 'adhb_pklnprt_q3-23_rev', 'adhb_pklnprt_q4-23_ril', 'adhb_pklnprt_q4-23_rev', 'adhb_pklnprt_2023_ril', 'adhb_pklnprt_2023_rev', 'adhb_pkp_q1-23_ril', 'adhb_pkp_q1-23_rev', 'adhb_pkp_q2-23_ril', 'adhb_pkp_q2-23_rev', 'adhb_pkp_q3-23_ril', 'adhb_pkp_q3-23_rev', 'adhb_pkp_q4-23_ril', 'adhb_pkp_q4-23_rev', 'adhb_pkp_2023_ril', 'adhb_pkp_2023_rev', 'adhb_pmtb_q1-23_ril', 'adhb_pmtb_q1-23_rev', 'adhb_pmtb_q2-23_ril'

#### 3. Evaluasi revisi distribusi

In [28]:
# 1. Ambil semua kolom distribusi 
revisi_dist_cols = [col for col in df.columns if col.startswith("distribusi")]

# 2. Buat base name tanpa suffix `ril/rev`
pairs = {}
for col in revisi_dist_cols:
    if "ril" in col or "rev" in col:
        base = "_".join(col.split("_")[:-1])  # buang suffix ril/rev
        pairs.setdefault(base, []).append(col)

# 3. Evaluasi distribusi
evaluasi_revisi_dist = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if c.endswith("_ril")][0]
        col1 = [c for c in cols if c.endswith("_rev")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df.index:
            v0 = df.at[idx, col0]
            v1 = df.at[idx, col1]

            if pd.notna(v0) and pd.notna(v1):
                selisih = abs(v1 - v0)
                if v0 * v1 < 0:
                    record = {
                        "kode": df.at[idx, "kode"],
                        "kabupaten/kota": df.at[idx, "kabupaten/kota"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Nilai revisi dan nilai rilis beda arah"
                    }
                    evaluasi_revisi_dist.append(record)
                elif selisih > 5:
                    record = {
                        "kode": df.at[idx, "kode"],
                        "kabupaten/kota": df.at[idx, "kabupaten/kota"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Nilai revisi lebih dari (+-5 point) dari nilai rilis"
                    }
                    evaluasi_revisi_dist.append(record)

# 4. Jadi DataFrame
evaluasi_revisi_dist = pd.DataFrame(evaluasi_revisi_dist)
print(evaluasi_revisi_dist)

    kode                        kabupaten/kota periode      nilai  \
0   1612  Kabupaten Penukal Abab Lematang Ilir   q1-23   0.000000   
1   1612  Kabupaten Penukal Abab Lematang Ilir   q2-23   0.000000   
2   1674                     Kota Lubuklinggau   q3-23   0.000000   
3   1674                     Kota Lubuklinggau   q4-23   0.000000   
4   1674                     Kota Lubuklinggau    2023   0.000000   
5   1612  Kabupaten Penukal Abab Lematang Ilir   q1-23   0.000000   
6   1612  Kabupaten Penukal Abab Lematang Ilir   q2-23   0.000000   
7   1674                     Kota Lubuklinggau   q3-23   0.000000   
8   1674                     Kota Lubuklinggau   q4-23   0.000000   
9   9608                 Kabupaten Puncak Jaya   q4-23  38.002616   
10  1674                     Kota Lubuklinggau    2023   0.000000   
11  1674                     Kota Lubuklinggau   q4-23   0.000000   
12  9608                 Kabupaten Puncak Jaya   q4-23  44.592365   
13  1612  Kabupaten Penukal Abab L

#### 4. Evaluasi revisi growth Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, Implisit Y-on-Y, SOG Q-to-Q, dan SOG Y-on-Y 

In [29]:
object_cols = df_clean.select_dtypes(include=["object"]).columns.tolist()

# 1. Ambil semua kolom Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, Implisit Y-on-Y, SOG Q-to-Q, dan SOG Y-on-Y 
revisi_growth_cols = [col for col in df_clean.columns if col.startswith("q-to-q") or col.startswith("y-on-y") or col.startswith("c-to-c") or col.startswith("implisit_q-to-q") or col.startswith("implisit_y-on-y") or col.startswith("sog_q") or col.startswith("sog_y-on-y")]

# 2. Buat base name tanpa suffix `ril/rev`
pairs = {}
for col in revisi_growth_cols:
    if "ril" in col or "rev" in col:
        base = "_".join(col.split("_")[:-1])  # buang suffix ril/rev
        pairs.setdefault(base, []).append(col)

# 3. Evaluasi growth
evaluasi_revisi_growth = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if c.endswith("_ril")][0]
        col1 = [c for c in cols if c.endswith("_rev")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df_clean.index:
            v0 = df_clean.at[idx, col0]
            v1 = df_clean.at[idx, col1]

            # Growth Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y
            if ('sog' not in col0 and any(x in col0 for x in ['q-to-q', 'y-on-y', 'c-to-c'])):
                # PDRB
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' in col0:
                    if type(v0) == str or type(v1) == str:
                        print("ini nilai v0:", v0, "field:", col0, "Type datanya:", type(v0))
                        print("ini nilai v1:", v1, "field:", col1, "Type datanya:", type(v1)) 

                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai revisi dan nilai rilis beda arah"
                        }
                        evaluasi_revisi_growth.append(record)
                    elif selisih > 3:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai revisi lebih dari (+-3 point) dari nilai rilis"
                        }
                        evaluasi_revisi_growth.append(record)
                # Komponen
                if pd.notna(v0) and pd.notna(v1) and 'pdrb' not in col0:
                    if type(v0) == str or type(v1) == str:
                        print("ini nilai v0:", v0, "field:", col0, "Type datanya:", type(v0))
                        print("ini nilai v1:", v1, "field:", col1, "Type datanya:", type(v1)) 

                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai revisi dan nilai rilis beda arah"
                        }
                        evaluasi_revisi_growth.append(record)
                    elif selisih > 4:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai revisi lebih dari (+-4 point) dari nilai rilis"
                        }
                        evaluasi_revisi_growth.append(record)

            # SoG PI (Q-to-Q, Y-on-Y)
            elif any(x in col0 for x in ['sog_q', 'sog_y-on-y']):
                if pd.notna(v0) and pd.notna(v1):
                    if type(v0) == str or type(v1) == str:
                        print("ini nilai v0:", v0, "field:", col0, "Type datanya:", type(v0))
                        print("ini nilai v1:", v1, "field:", col1, "Type datanya:", type(v1)) 

                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Nilai revisi dan nilai rilis beda arah"
                        }
                        evaluasi_revisi_growth.append(record)
                    elif 'pi' in col0:
                        if v1 < -2 or v1 > 2:
                            record = {
                                "kode": df_clean.at[idx, "kode"],
                                "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                                "periode": periode,
                                "nilai": v1,
                                "kolom": col0 + " vs " + col1,
                                "evaluasi": "SoG PI lebih dari +-2 persen"
                            }
                            evaluasi_revisi_growth.append(record)

# 4. Jadi DataFrame
evaluasi_revisi_growth = pd.DataFrame(evaluasi_revisi_growth)
print(evaluasi_revisi_growth)

      kode                  kabupaten/kota periode      nilai  \
0     9706                Kabupaten Yalimo   q1-23  -0.355423   
1     1674               Kota Lubuklinggau   q3-23   0.000000   
2     9503                 Kabupaten Mappi   q3-23  21.190028   
3     6101                Kabupaten Sambas   q1-23   0.109321   
4     7412      Kabupaten Konawe Kepulauan   q4-23   2.785376   
...    ...                             ...     ...        ...   
1162  1507  Kabupaten Tanjung Jabung Barat    2023  -0.170630   
1163  1509                 Kabupaten Bungo    2023  -0.079260   
1164  3203               Kabupaten Cianjur    2023   0.082748   
1165  6303                Kabupaten Banjar    2023   0.403986   
1166  7204                  Kabupaten Poso    2023  -0.023897   

                                                  kolom  \
0        q-to-q_pdrb_q1-23_ril vs q-to-q_pdrb_q1-23_rev   
1        q-to-q_pdrb_q3-23_ril vs q-to-q_pdrb_q3-23_rev   
2        q-to-q_pdrb_q3-23_ril vs q-to-q_p

#### 5. Evaluasi growth Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, Implisit Y-on-Y, SOG Q-to-Q, dan SOG Y-on-Y 

In [30]:
object_cols = df_clean.select_dtypes(include=["object"]).columns.tolist()

# 1. Ambil semua kolom Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, Implisit Y-on-Y, SOG Q-to-Q, dan SOG Y-on-Y 
value_growth_cols = [col for col in df_clean.columns if col.startswith("q-to-q") or col.startswith("y-on-y") or col.startswith("c-to-c") or col.startswith("implisit_q-to-q") or col.startswith("implisit_y-on-y") or col.startswith("sog_q") or col.startswith("sog_y-on-y")]

# 2. Buat base name tanpa suffix `ril/rev`
pairs = {}
for col in value_growth_cols:
    if "prov" in col or "kabkot" in col:
        base = "_".join(col.split("_")[:-1])  # buang suffix ril/rev
        pairs.setdefault(base, []).append(col)

# 3. Evaluasi growth prov vs kabkot dan triwulan berjalan
evaluasi_value_growth = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if c.endswith("_prov")][0]
        col1 = [c for c in cols if c.endswith("_kabkot")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df_clean.index:
            v0 = df_clean.at[idx, col0]
            v1 = df_clean.at[idx, col1]

            # Growth Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y
            if ('sog' not in col0 and any(x in col0 for x in ['q-to-q', 'y-on-y', 'c-to-c'])):
                if pd.notna(v0) and pd.notna(v1):
                    if type(v0) == str or type(v1) == str:
                        print("ini nilai v0:", v0, "field:", col0, "Type datanya:", type(v0))
                        print("ini nilai v1:", v1, "field:", col1, "Type datanya:", type(v1)) 

                    selisih = abs(v1 - v0)
                    if v0 * v1 < 0:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Beda arah dengan nilai provinsi"
                        }
                        evaluasi_value_growth.append(record)
                    elif selisih > 4:
                        record = {
                            "kode": df_clean.at[idx, "kode"],
                            "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                            "periode": periode,
                            "nilai": v1,
                            "kolom": col0 + " vs " + col1,
                            "evaluasi": "Selisih dengan nilai provinsi cukup jauh (+-4 point)"
                        }
                        evaluasi_value_growth.append(record)

unpaired_columns = get_unpaired_columns(value_growth_cols)
for col in unpaired_columns:
    for idx, val in df_clean[col].items():
        # PI
        if "pi" in col:
            if pd.notna(val) and (val < -2 or val > 2):
                print(col, idx, val)
                record = {
                    "kode": df_clean.at[idx, "kode"],
                    "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                    "periode": periode,
                    "nilai": val,
                    "kolom": col,
                    "evaluasi": "SoG PI lebih dari +-2 persen"
                }
                evaluasi_value_growth.append(record)

# 4. Jadi DataFrame
evaluasi_value_growth = pd.DataFrame(evaluasi_value_growth)
print(evaluasi_value_growth)

Empty DataFrame
Columns: []
Index: []


## Menggabungkan seluruh hasil evaluasi

In [31]:
evaluasi = concat_evaluasi([evaluasi_revisi_harga, evaluasi_revisi_dist, evaluasi_revisi_growth, evaluasi_value_growth])
print(evaluasi.head())

   kode       kabupaten/kota periode          nilai  \
0  1502   Kabupaten Merangin   q1-23   -1179.636037   
1  2101    Kabupaten Karimun   q1-23   94105.611962   
2  2102     Kabupaten Bintan   q1-23   83787.816676   
3  2172  Kota Tanjung Pinang   q1-23  124412.849566   
4  7604     Kabupaten Mamuju   q1-23  206001.034483   

                                    kolom  \
0  adhb_pi_q1-23_ril vs adhb_pi_q1-23_rev   
1  adhb_pi_q1-23_ril vs adhb_pi_q1-23_rev   
2  adhb_pi_q1-23_ril vs adhb_pi_q1-23_rev   
3  adhb_pi_q1-23_ril vs adhb_pi_q1-23_rev   
4  adhb_pi_q1-23_ril vs adhb_pi_q1-23_rev   

                                           evaluasi  
0  Nilai revisi lebih dari (+-30%) dari nilai rilis  
1  Nilai revisi lebih dari (+-30%) dari nilai rilis  
2  Nilai revisi lebih dari (+-30%) dari nilai rilis  
3  Nilai revisi lebih dari (+-30%) dari nilai rilis  
4  Nilai revisi lebih dari (+-30%) dari nilai rilis  


## Menuliskan hasil evaluasi kedalam file excel

In [32]:
write_to_excel(evaluasi, "kabupaten/kota")

Sheet 1502 ditambahkan ke data/output/evaluasi_15.xlsx
Sheet 2101 ditambahkan ke data/output/evaluasi_21.xlsx
Sheet 2102 ditambahkan ke data/output/evaluasi_21.xlsx
Sheet 2172 ditambahkan ke data/output/evaluasi_21.xlsx
Sheet 7604 ditambahkan ke data/output/evaluasi_76.xlsx
Sheet 7605 ditambahkan ke data/output/evaluasi_76.xlsx
Sheet 1507 ditambahkan ke data/output/evaluasi_15.xlsx
Sheet 7601 ditambahkan ke data/output/evaluasi_76.xlsx
Sheet 6206 ditambahkan ke data/output/evaluasi_62.xlsx
Sheet 6303 ditambahkan ke data/output/evaluasi_63.xlsx
Sheet 9103 ditambahkan ke data/output/evaluasi_91.xlsx
Sheet 9104 ditambahkan ke data/output/evaluasi_91.xlsx
Sheet 9111 ditambahkan ke data/output/evaluasi_91.xlsx
Sheet 1605 ditambahkan ke data/output/evaluasi_16.xlsx
Sheet 1807 ditambahkan ke data/output/evaluasi_18.xlsx
Sheet 3172 ditambahkan ke data/output/evaluasi_31.xlsx
Sheet 3173 ditambahkan ke data/output/evaluasi_31.xlsx
Sheet 3174 ditambahkan ke data/output/evaluasi_31.xlsx
Sheet 3175